In [157]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import FunctionTransformer
import pandas as pd
import numpy as np
import joblib

In [158]:
df = pd.read_csv("TehranHouse.csv")
print(df.columns[df.isnull().any()].tolist())
df = df.dropna(subset=["Address"])

['Address']


In [159]:
df.info()

<class 'pandas.DataFrame'>
Index: 3456 entries, 0 to 3478
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Area        3456 non-null   str    
 1   Room        3456 non-null   int64  
 2   Parking     3456 non-null   bool   
 3   Warehouse   3456 non-null   bool   
 4   Elevator    3456 non-null   bool   
 5   Address     3456 non-null   str    
 6   Price       3456 non-null   float64
 7   Price(USD)  3456 non-null   float64
dtypes: bool(3), float64(2), int64(1), str(2)
memory usage: 172.1 KB


In [160]:
df.isna().sum()

Area          0
Room          0
Parking       0
Warehouse     0
Elevator      0
Address       0
Price         0
Price(USD)    0
dtype: int64

In [161]:
df.head()

,Area,Room,Parking,Warehouse,Elevator,Address,Price,Price(USD)
0,63,1,True,True,True,Shahran,1.850000e+09,61666.67
1,60,1,True,True,True,Shahran,1.850000e+09,61666.67
2,79,2,True,True,True,Pardis,5.500000e+08,18333.33
3,95,2,True,True,True,Shahrake Qods,9.025000e+08,30083.33
4,123,2,True,True,True,Shahrake Gharb,7.000000e+09,233333.33


In [162]:
print(df.duplicated().sum())
df = df.drop_duplicates()

208


In [163]:
df.Address.value_counts()

Address
Punak                     148
West Ferdows Boulevard    133
Gheitarieh                133
Shahran                   123
Pardis                    123
                         ... 
Firoozkooh                  1
Shadabad                    1
Naziabad                    1
Javadiyeh                   1
Yakhchiabad                 1
Name: count, Length: 192, dtype: int64

In [164]:
print(df.Room.unique())
print(df.Area.unique())

[1 2 3 0 4 5]
<StringArray>
[             '63',              '60',              '79',              '95',
             '123',              '70',              '87',              '59',
              '54',              '71',
 ...
 ' 2,550,000,000 ',             '330',             '212',             '345',
             '530',             '630',              '30',             '205',
             '285',             '186']
Length: 243, dtype: str


Clean the Area Column :
Convert `Area` to numeric values. Anything that cannot be converted becomes NaN and is then dropped.

In [165]:
def convert_area_to_num(x):
    try:
        return float(x)
    except:
        return None

In [166]:
df2 = df.copy()
df2["Area"] = df2["Area"].apply(convert_area_to_num)

In [167]:
df2[df2["Area"]>500]

,Area,Room,Parking,Warehouse,Elevator,Address,Price,Price(USD)
356,650.0,2,True,True,False,Absard,5.200000e+09,173333.33
431,660.0,5,True,True,False,Lavasan,5.500000e+10,1833333.33
573,863.0,2,True,True,True,Gheitarieh,7.830000e+09,261000.00
819,680.0,5,True,True,False,Ekhtiarieh,8.160000e+10,2720000.00
831,750.0,5,True,True,True,Mahmoudieh,7.500000e+10,2500000.00
1236,600.0,2,True,True,True,Qalandari,9.150000e+09,305000.00
1407,574.0,5,True,True,False,North Program Organization,6.000000e+10,2000000.00
1694,929.0,5,True,True,False,Zafar,8.000000e+10,2666666.67
1700,550.0,3,True,True,False,Absard,4.500000e+09,150000.00
1810,705.0,5,True,True,False,Abazar,9.100000e+10,3033333.33


In [168]:
df2 = df2.dropna()

In [169]:
df2.isna().sum()

Area          0
Room          0
Parking       0
Warehouse     0
Elevator      0
Address       0
Price         0
Price(USD)    0
dtype: int64

In [170]:
df2["Area"].describe()

count    3242.000000
mean      107.701110
std        70.935755
min        30.000000
25%        70.000000
50%        90.000000
75%       121.000000
max       929.000000
Name: Area, dtype: float64

In [171]:
ratio = df2["Price"] / df2["Price(USD)"]
print(ratio.describe())
print(ratio.round(0).value_counts().head(10))

count     3242.000000
mean     29999.999953
std          0.002403
min      29999.972727
25%      29999.999547
50%      30000.000000
75%      30000.000395
max      30000.054546
dtype: float64
30000.0    3242
Name: count, dtype: int64


In [172]:
df2["price_per_m"] = df2["Price(USD)"] / df2["Area"]
print(df2["price_per_m"].describe())
print(df2.sort_values("price_per_m").head(10))
print(df2.sort_values("price_per_m").tail(10))

count     3242.000000
mean      1389.548324
std       1064.418597
min          0.750000
25%        666.666727
50%       1166.666658
75%       1833.333352
max      13888.888889
Name: price_per_m, dtype: float64
       Area  Room  Parking  Warehouse  Elevator                    Address  \
136   160.0     1    False      False     False                    Qarchak   
2770   83.0     2     True       True      True                      Ozgol   
731    75.0     2     True       True      True                     Pardis   
2721  110.0     0     True       True      True                     Parand   
2201   49.0     1     True       True     False                   Andisheh   
3300  120.0     3     True       True      True  Persian Gulf Martyrs Lake   
1343   78.0     2     True       True      True                     Parand   
3149  118.0     2     True       True      True           Northern Chitgar   
1443  350.0     3     True       True     False                   Pasdaran   
1543  123.

In [173]:
df2[df2.Area/df2.Room<28].head()

,Area,Room,Parking,Warehouse,Elevator,Address,Price,Price(USD),price_per_m
8,54.0,2,True,True,False,Andisheh,4.930000e+08,16433.33,304.320926
776,54.0,2,True,True,False,Shahran,1.850000e+09,61666.67,1141.975370
846,51.0,2,False,True,True,Komeil,1.050000e+09,35000.00,686.274510
1371,55.0,2,False,False,False,Parastar,1.280000e+09,42666.67,775.757636
1929,50.0,2,True,True,False,Shahran,1.820000e+09,60666.67,1213.333400


In [174]:
df2.shape

(3242, 9)

Remove Unrealistic Area-per-Room Rows:
Drop rows where the area per room is below 28 m², which are unrealistic (e.g. 54 m² with 2 rooms is borderline, 160 m² for a 0.75 $/m² listing is clearly wrong).


In [175]:
df3 = df2[~(df2.Area/df2.Room<28)]

In [176]:
df3.shape

(3214, 9)

Remove Price-per-Meter Outliers per Neighborhood :
For each neighborhood, keep only rows whose price per m² is within `k` standard deviations of that neighborhood's mean. Neighborhoods are handled separately because prices differ greatly between areas.

In [177]:
def remove_ppm_outliers(df, k=2):
    g = df.groupby("Address")["price_per_m"]
    m = g.transform("mean")
    st = g.transform("std").fillna(np.inf)
    return df[(df.price_per_m > m - k*st) & (df.price_per_m <= m + k*st)]

In [178]:
train_df, test_df = train_test_split(df3, test_size=0.2, random_state=42)
train_df = remove_ppm_outliers(train_df)

min_count = 5
counts = train_df["Address"].value_counts()
rare = counts[counts < min_count].index

train_df = train_df.copy()
test_df = test_df.copy()
train_df["Address"] = train_df["Address"].replace(rare, "other")
test_df["Address"] = test_df["Address"].replace(rare, "other")

drop_cols = ["Price", "Price(USD)", "price_per_m"]
X_train = train_df.drop(columns=drop_cols)
y_train = train_df["Price(USD)"]
X_test = test_df.drop(columns=drop_cols)
y_test = test_df["Price(USD)"]

In [179]:
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["Address"]),
    ("scaler", StandardScaler(), ["Room"]),
    ("area", Pipeline([("log", FunctionTransformer(np.log1p)),
                       ("sc", StandardScaler())]), ["Area"]),
], remainder="passthrough")

Models and Hyperparameter Grids :
Define the candidate models (SVR, Random Forest, Ridge, Decision Tree) and the hyperparameters to search for each.

In [180]:
model_params = {
    "svr": {
        "model": SVR(),
        "params": {
            "C": [1, 10, 100],
            "kernel": ["rbf", "linear"]
        }
    },
    "random forest": {
        "model": RandomForestRegressor(random_state=42),
        "params": {
            "n_estimators": [50, 100, 200],
        }
    },
    "ridge": {
        "model": Ridge(),
        "params": {
            "alpha": [0.1, 1, 10, 100],
        }
    },
    "tree": {
        "model": DecisionTreeRegressor(random_state=42),
        "params": {
            "criterion": ["squared_error", "absolute_error"],
            "max_depth": [None, 5, 10, 20]
        }
    }
}

Train and Compare Models :
For each model, run GridSearchCV with 5-fold cross-validation inside a pipeline. The target is log-transformed (`log1p`) to reduce skew and converted back for predictions. Record the best CV score, best parameters, and test score.

In [181]:
scores = []

for model_name, mp in model_params.items():
    grid = GridSearchCV(mp["model"], mp["params"], cv=5)
    pipe = Pipeline([("pre", pre), ("grid", grid)])
    model = TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1)
    model.fit(X_train, y_train)
    fitted_grid = model.regressor_.named_steps["grid"]
    scores.append({
        "model": model_name,
        "best_score": fitted_grid.best_score_,
        "best_params": fitted_grid.best_params_,
        "test_score": model.score(X_test, y_test),
    })

In [182]:
scores

[{'model': 'svr',
  'best_score': np.float64(0.8887893273739035),
  'best_params': {'C': 1, 'kernel': 'linear'},
  'test_score': 0.7510254906243312},
 {'model': 'random forest',
  'best_score': np.float64(0.8425667811505588),
  'best_params': {'n_estimators': 200},
  'test_score': 0.695953953471126},
 {'model': 'ridge',
  'best_score': np.float64(0.8878364473371642),
  'best_params': {'alpha': 1},
  'test_score': 0.7472990151132834},
 {'model': 'tree',
  'best_score': np.float64(0.7956795275621307),
  'best_params': {'criterion': 'absolute_error', 'max_depth': 20},
  'test_score': 0.5824243521212741}]